# Recursive Language Model (RLM) Agent

Demonstrates three progressively more sophisticated implementations of the **RLM pattern** — an agent that analyzes long documents by recursively decomposing them with a hierarchy of models.

**Features covered:**
- V1: Simple ReAct agent with peek/regex/subcall tools using `create_agent`
- V2: Modular `RLMAgent` class with factory methods and `create_deep_agent()`
- V3: Custom LangGraph implementation with explicit state, nodes, and conditional edges
- V4: Minimal REPL-based RLM using code generation and `exec()`

**Prerequisites:**
- `deepagents`, `langchain-openai`, `langchain`, `python-dotenv` packages
- `OPENAI_API_KEY` environment variable (loaded from `~/.env/orchestra/.env.backend`)
- A `paper.txt` file in the working directory (for V4)

In [ ]:
!uv pip install -q langchain-openai langchain deepagents python-dotenv

## Environment Setup

Load API keys from the standard Orchestra env file.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from standard location
# ~/.env/orchestra/.env.backend
env_path = Path.home() / ".env" / "orchestra" / ".env.backend"
load_dotenv(env_path)

## V1 — Simple ReAct Agent with RLM Tools

Uses `create_agent` from LangChain to build a ReAct-style agent that explores a long document through **peek**, **find_regex**, **context_info**, and **subcall** tools. The root agent delegates focused analysis to a cheaper sub-agent.

In [ ]:
"""
-----------------------------------------------------------------------------
V1 of the RLM-Style ReAct Agent Demo
-----------------------------------------------------------------------------
RLM-Style ReAct Agent Demo for Google Colab

This demo uses the RLM blog post itself as the long context,
demonstrating the recursive language model pattern described in the paper.

Setup (run in Colab):
    !pip install langchain-openai langgraph requests

Then set your API key:
    import os
    os.environ["OPENAI_API_KEY"] = "your-key-here"
"""
from __future__ import annotations
import re
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# =============================================================================
# STEP 1: Fetch the RLM blog post as our "long context"
# =============================================================================

def fetch_rlm_blog() -> str:
    """Fetch and clean the RLM blog post content."""
    url = "https://alexzhang13.github.io/blog/2025/rlm/"
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    # Remove script and style elements
    for element in soup(['script', 'style', 'nav', 'header', 'footer']):
        element.decompose()

    # Get the main content
    main_content = soup.find('main') or soup.find('article') or soup.body

    if main_content:
        text = main_content.get_text(separator='\n', strip=True)
    else:
        text = soup.get_text(separator='\n', strip=True)

    # Clean up excessive whitespace
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return '\n'.join(lines)

# =============================================================================
# STEP 2: Initialize the environment (the long context)
# =============================================================================

print("Fetching RLM blog post as context...")
PROMPT = fetch_rlm_blog()
print(f"Context loaded: {len(PROMPT):,} characters")

# =============================================================================
# STEP 3: Define tools for interacting with the context
# =============================================================================

@tool
def peek(start: int, end: int) -> str:
    """
    Return a slice of the context: PROMPT[start:end].
    Use this to examine specific regions of the document.

    Args:
        start: Starting character index (0-based)
        end: Ending character index (exclusive)
    """
    start = max(0, start)
    end = max(start, min(len(PROMPT), end))
    snippet = PROMPT[start:end]
    return f"[Characters {start}-{end} of {len(PROMPT)} total]\n\n{snippet}"


@tool
def find_regex(pattern: str, max_hits: int = 15) -> str:
    """
    Search for a regex pattern in the context.
    Returns match positions with surrounding context for each hit.

    Args:
        pattern: Regular expression pattern to search for
        max_hits: Maximum number of matches to return (default 15)
    """
    hits = []
    try:
        for m in re.finditer(pattern, PROMPT, flags=re.MULTILINE | re.IGNORECASE):
            s, e = m.start(), m.end()
            # Get surrounding context
            context_start = max(0, s - 80)
            context_end = min(len(PROMPT), e + 80)
            preview = PROMPT[context_start:context_end].replace('\n', ' ')

            # Mark the match
            rel_start = s - context_start
            rel_end = e - context_start
            marked = f"...{preview[:rel_start]}>>>{preview[rel_start:rel_end]}<<<{preview[rel_end:]}..."

            hits.append(f"[{s}:{e}] {marked}")
            if len(hits) >= max_hits:
                break
    except re.error as err:
        return f"REGEX_ERROR: {err}"

    return '\n\n'.join(hits) if hits else "NO_MATCHES"


@tool
def context_info() -> str:
    """
    Get basic information about the context document.
    Returns total length, line count, and a structural overview.
    """
    lines = PROMPT.split('\n')

    # Find section headers (lines that look like titles)
    sections = []
    for i, line in enumerate(lines):
        # Heuristic: short lines that might be headers
        if 3 < len(line) < 100 and not line.endswith('.') and not line.startswith('-'):
            if any(keyword in line.lower() for keyword in
                   ['result', 'model', 'conclusion', 'introduction', 'method',
                    'experiment', 'related', 'discussion', 'limitation', 'rlm',
                    'recursive', 'benchmark', 'setup', 'acknowledgement']):
                sections.append(f"  Line {i}: {line[:60]}...")

    return (
        f"Document Statistics:\n"
        f"  Total characters: {len(PROMPT):,}\n"
        f"  Total lines: {len(lines):,}\n"
        f"\nPotential section headers found:\n" + '\n'.join(sections[:20])
    )


# =============================================================================
# STEP 4: Set up the models (root + cheaper sub-agent)
# =============================================================================
SMART = "gpt-4.1-mini"
FAST = "gpt-4.1-nano"

# Root model: more capable, orchestrates the analysis
root_llm = ChatOpenAI(model=SMART, temperature=0)

# Sub model: cheaper, handles delegated sub-queries
sub_llm = ChatOpenAI(model=FAST, temperature=0)

# Create sub-agent (no subcall tool - prevents infinite recursion)
sub_agent = create_agent(
    model=sub_llm,
    tools=[peek, find_regex, context_info],
)


def _invoke_subagent(query: str) -> str:
    """Invoke the sub-agent and extract its final response."""
    result = sub_agent.invoke({"messages": [("user", query)]})
    # Get the last AI message content
    for msg in reversed(result["messages"]):
        if hasattr(msg, 'content') and msg.content and msg.type == "ai":
            return msg.content
    return "Sub-agent produced no response"


@tool
def subcall(snippet_start: int, snippet_end: int, question: str) -> str:
    """
    Delegate analysis of a specific region to a sub-agent (cheaper model).
    Use this for focused analysis of document sections.

    The sub-agent has access to peek, find_regex, and context_info tools.

    Args:
        snippet_start: Start index of the region to analyze
        snippet_end: End index of the region to analyze
        question: The specific question to answer about this region
    """
    # Bound the indices
    snippet_start = max(0, snippet_start)
    snippet_end = min(len(PROMPT), snippet_end)
    snippet = PROMPT[snippet_start:snippet_end]

    sub_query = (
        f"You are analyzing a snippet from a larger document about Recursive Language Models.\n\n"
        f"SNIPPET (characters {snippet_start}-{snippet_end} of {len(PROMPT)} total):\n"
        f"```\n{snippet}\n```\n\n"
        f"QUESTION: {question}\n\n"
        f"Provide a focused, concise answer based on this snippet."
    )

    return _invoke_subagent(sub_query)


# =============================================================================
# STEP 5: Create the root agent
# =============================================================================

root_agent = create_agent(
    model=root_llm,
    tools=[peek, find_regex, context_info, subcall],
)

# System message that primes the agent for RLM-style behavior
SYSTEM_MESSAGE = f"""You are an RLM (Recursive Language Model) analyzing a document about RLMs - how meta!

CRITICAL: You CANNOT see the document directly. It is stored in an environment variable.
The document has {len(PROMPT):,} characters total.

Your available tools:
- context_info(): Get document statistics and find section headers
- peek(start, end): View a slice of the document by character indices
- find_regex(pattern): Search for patterns and see matches with context
- subcall(start, end, question): Delegate focused analysis of a region to a cheaper sub-agent

Strategy tips:
1. Start with context_info() to understand the document structure
2. Use find_regex() to locate relevant sections by keywords
3. Use peek() to read specific regions you've identified
4. Use subcall() to delegate detailed analysis of specific sections

Build your answer incrementally. Be systematic and thorough."""


# =============================================================================
# STEP 6: Demo queries
# =============================================================================

DEMO_QUERIES = {
    "benchmarks": (
        "What benchmarks were used to evaluate RLMs? For each benchmark, explain "
        "what it tests and summarize the key results comparing RLMs to baselines."
    ),

    "strategies": (
        "What emergent strategies did the authors observe RLMs using when analyzing context? "
        "List each strategy with a brief description of how it works."
    ),

    "limitations": (
        "What are the stated limitations of RLMs according to this paper? "
        "Also identify any implicit limitations you can infer from the methodology."
    ),

    "vs_agents": (
        "How do RLMs differ from traditional agent architectures like ReAct? "
        "What is the key philosophical difference in how they approach context decomposition?"
    ),

    "cost_analysis": (
        "Analyze the cost-effectiveness of RLMs based on the experimental results. "
        "Compare API costs between RLM approaches and baseline models."
    ),

    "future_work": (
        "What future directions do the authors suggest for RLM research? "
        "What aspects do they think could be improved or extended?"
    ),
}


def run_query(query_name: str = "benchmarks", verbose: bool = True):
    """
    Run a demo query against the RLM blog post.

    Args:
        query_name: One of: benchmarks, strategies, limitations, vs_agents, cost_analysis, future_work
        verbose: If True, prints intermediate steps
    """
    if query_name not in DEMO_QUERIES:
        print(f"Unknown query. Choose from: {list(DEMO_QUERIES.keys())}")
        return

    question = DEMO_QUERIES[query_name]
    print(f"\n{'='*60}")
    print(f"QUERY: {query_name}")
    print(f"{'='*60}")
    print(f"\n{question}\n")
    print(f"{'='*60}\n")

    messages = [
        ("system", SYSTEM_MESSAGE),
        ("user", question)
    ]

    for chunk in root_agent.stream(
        {"messages": messages},
        stream_mode="values",
    ):
        if "messages" in chunk and verbose:
            msg = chunk["messages"][-1]
            if hasattr(msg, 'pretty_print'):
                msg.pretty_print()
            elif hasattr(msg, 'content'):
                print(f"\n{msg.type}: {msg.content[:500]}..." if len(str(msg.content)) > 500 else f"\n{msg.type}: {msg.content}")


def run_custom_query(question: str, verbose: bool = True):
    """
    Run a custom query against the RLM blog post.

    Args:
        question: Your question about the RLM paper
        verbose: If True, prints intermediate steps
    """
    print(f"\n{'='*60}")
    print(f"CUSTOM QUERY")
    print(f"{'='*60}")
    print(f"\n{question}\n")
    print(f"{'='*60}\n")

    messages = [
        ("system", SYSTEM_MESSAGE),
        ("user", question)
    ]

    final_response = None
    for chunk in root_agent.stream(
        {"messages": messages},
        stream_mode="values",
    ):
        if "messages" in chunk:
            msg = chunk["messages"][-1]
            if verbose and hasattr(msg, 'pretty_print'):
                msg.pretty_print()
            if hasattr(msg, 'content') and msg.type == "ai":
                final_response = msg.content

    return final_response


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    print("\n" + "="*60)
    print("RLM Demo Ready!")
    print("="*60)
    print(f"\nContext loaded: {len(PROMPT):,} characters from the RLM blog post")
    print("\nAvailable demo queries:")
    for name, q in DEMO_QUERIES.items():
        print(f"  - {name}: {q[:60]}...")

    print("\nUsage:")
    print("  run_query('benchmarks')           # Run a preset query")
    print("  run_query('strategies')           # Try different presets")
    print("  run_custom_query('Your question') # Ask anything")

    print("\n" + "-"*60)
    print("Running demo query: 'benchmarks'")
    print("-"*60)

    run_query("benchmarks")

## V2 — Modular RLMAgent Class

A reusable `RLMAgent` class with factory methods (`from_url`, `from_file`, `from_text`). Uses `create_deep_agent()` internally for both the root and sub-agent, with the `name` parameter set for trace identification.

In [ ]:
"""
-----------------------------------------------------------------------------
V2 of the RLM-Style ReAct Agent Demo
-----------------------------------------------------------------------------
RLM-Style ReAct Agent - Modular Implementation

A reusable framework for creating Recursive Language Model agents that can
analyze long documents using a hierarchy of models (smart root + cheap sub-agents).

Usage:
    from rlm_agent import RLMAgent, ContentFetcher

    # Option 1: From URL
    agent = RLMAgent.from_url("https://example.com/article")

    # Option 2: From text
    agent = RLMAgent.from_text("Your long document here...")

    # Option 3: From file
    agent = RLMAgent.from_file("document.txt")

    # Run queries
    response = agent.query("What are the main findings?")

    # Stream responses
    for chunk in agent.stream("Summarize the methodology"):
        print(chunk, end="", flush=True)
"""
from __future__ import annotations

import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterator, Callable

import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class ModelConfig:
    """Configuration for the LLM models used by the agent."""
    root_model: str = "gpt-4.1-mini"
    sub_model: str = "gpt-4.1-nano"
    temperature: float = 0.0


@dataclass
class AgentConfig:
    """Configuration for agent behavior."""
    max_regex_hits: int = 15
    peek_context_chars: int = 80
    model_config: ModelConfig = field(default_factory=ModelConfig)


# =============================================================================
# Content Fetching
# =============================================================================

class ContentFetcher(ABC):
    """Abstract base class for content fetching strategies."""

    @abstractmethod
    def fetch(self) -> str:
        """Fetch and return the content as a string."""
        pass


class URLFetcher(ContentFetcher):
    """Fetch content from a URL and extract text."""

    def __init__(self, url: str, timeout: int = 30):
        self.url = url
        self.timeout = timeout

    def fetch(self) -> str:
        response = requests.get(self.url, timeout=self.timeout)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove non-content elements
        for element in soup(['script', 'style', 'nav', 'header', 'footer']):
            element.decompose()

        # Find main content
        main_content = soup.find('main') or soup.find('article') or soup.body

        if main_content:
            text = main_content.get_text(separator='\n', strip=True)
        else:
            text = soup.get_text(separator='\n', strip=True)

        # Clean up whitespace
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        return '\n'.join(lines)


class FileFetcher(ContentFetcher):
    """Fetch content from a local file."""

    def __init__(self, path: str | Path, encoding: str = "utf-8"):
        self.path = Path(path)
        self.encoding = encoding

    def fetch(self) -> str:
        return self.path.read_text(encoding=self.encoding)


class TextFetcher(ContentFetcher):
    """Pass-through fetcher for raw text."""

    def __init__(self, text: str):
        self.text = text

    def fetch(self) -> str:
        return self.text


# =============================================================================
# Tool Factory
# =============================================================================

class ToolFactory:
    """Creates tools bound to a specific context document."""

    def __init__(self, context: str, config: AgentConfig):
        self.context = context
        self.config = config

    def create_peek_tool(self) -> Callable:
        """Create a peek tool bound to this context."""
        context = self.context

        @tool
        def peek(start: int, end: int) -> str:
            """
            Return a slice of the context: PROMPT[start:end].
            Use this to examine specific regions of the document.

            Args:
                start: Starting character index (0-based)
                end: Ending character index (exclusive)
            """
            start = max(0, start)
            end = max(start, min(len(context), end))
            snippet = context[start:end]
            return f"[Characters {start}-{end} of {len(context)} total]\n\n{snippet}"

        return peek

    def create_find_regex_tool(self) -> Callable:
        """Create a regex search tool bound to this context."""
        context = self.context
        max_hits = self.config.max_regex_hits
        peek_chars = self.config.peek_context_chars

        @tool
        def find_regex(pattern: str, max_results: int | None = None) -> str:
            """
            Search for a regex pattern in the context.
            Returns match positions with surrounding context for each hit.

            Args:
                pattern: Regular expression pattern to search for
                max_results: Maximum number of matches to return (default from config)
            """
            limit = max_results or max_hits
            hits = []

            try:
                for m in re.finditer(pattern, context, flags=re.MULTILINE | re.IGNORECASE):
                    s, e = m.start(), m.end()
                    context_start = max(0, s - peek_chars)
                    context_end = min(len(context), e + peek_chars)
                    preview = context[context_start:context_end].replace('\n', ' ')

                    rel_start = s - context_start
                    rel_end = e - context_start
                    marked = f"...{preview[:rel_start]}>>>{preview[rel_start:rel_end]}<<<{preview[rel_end:]}..."

                    hits.append(f"[{s}:{e}] {marked}")
                    if len(hits) >= limit:
                        break
            except re.error as err:
                return f"REGEX_ERROR: {err}"

            return '\n\n'.join(hits) if hits else "NO_MATCHES"

        return find_regex

    def create_context_info_tool(self) -> Callable:
        """Create a context info tool bound to this context."""
        context = self.context

        @tool
        def context_info() -> str:
            """
            Get basic information about the context document.
            Returns total length, line count, and a structural overview.
            """
            lines = context.split('\n')

            sections = []
            keywords = [
                'result', 'model', 'conclusion', 'introduction', 'method',
                'experiment', 'related', 'discussion', 'limitation', 'rlm',
                'recursive', 'benchmark', 'setup', 'acknowledgement', 'abstract',
                'summary', 'overview', 'background', 'analysis', 'evaluation'
            ]

            for i, line in enumerate(lines):
                if 3 < len(line) < 100 and not line.endswith('.') and not line.startswith('-'):
                    if any(kw in line.lower() for kw in keywords):
                        sections.append(f"  Line {i}: {line[:60]}...")

            return (
                f"Document Statistics:\n"
                f"  Total characters: {len(context):,}\n"
                f"  Total lines: {len(lines):,}\n"
                f"\nPotential section headers found:\n" + '\n'.join(sections[:20])
            )

        return context_info

    def create_subcall_tool(self, sub_agent) -> Callable:
        """Create a subcall tool that delegates to a sub-agent."""
        context = self.context

        def invoke_subagent(query: str) -> str:
            result = sub_agent.invoke({"messages": [("user", query)]})
            for msg in reversed(result["messages"]):
                if hasattr(msg, 'content') and msg.content and msg.type == "ai":
                    return msg.content
            return "Sub-agent produced no response"

        @tool
        def subcall(snippet_start: int, snippet_end: int, question: str) -> str:
            """
            Delegate analysis of a specific region to a sub-agent (cheaper model).
            Use this for focused analysis of document sections.

            The sub-agent has access to peek, find_regex, and context_info tools.

            Args:
                snippet_start: Start index of the region to analyze
                snippet_end: End index of the region to analyze
                question: The specific question to answer about this region
            """
            snippet_start = max(0, snippet_start)
            snippet_end = min(len(context), snippet_end)
            snippet = context[snippet_start:snippet_end]

            sub_query = (
                f"You are analyzing a snippet from a larger document.\n\n"
                f"SNIPPET (characters {snippet_start}-{snippet_end} of {len(context)} total):\n"
                f"```\n{snippet}\n```\n\n"
                f"QUESTION: {question}\n\n"
                f"Provide a focused, concise answer based on this snippet."
            )

            return invoke_subagent(sub_query)

        return subcall

    def create_base_tools(self) -> list[Callable]:
        """Create the base tools (without subcall)."""
        return [
            self.create_peek_tool(),
            self.create_find_regex_tool(),
            self.create_context_info_tool(),
        ]


# =============================================================================
# Main Agent Class
# =============================================================================

class RLMAgent:
    """
    A Recursive Language Model agent for analyzing long documents.

    Uses a hierarchy of models:
    - Root model (smarter): orchestrates analysis, delegates tasks
    - Sub model (cheaper): handles focused sub-queries on document sections
    """

    def __init__(
        self,
        context: str,
        config: AgentConfig | None = None,
        system_message: str | None = None,
        document_description: str = "a document",
    ):
        """
        Initialize the RLM agent.

        Args:
            context: The full document text to analyze
            config: Agent configuration (uses defaults if None)
            system_message: Custom system message (auto-generated if None)
            document_description: Description of the document for the system prompt
        """
        self.context = context
        self.config = config or AgentConfig()
        self.document_description = document_description

        # Create tool factory
        self.tool_factory = ToolFactory(context, self.config)

        # Initialize models
        model_cfg = self.config.model_config
        self.root_llm = ChatOpenAI(
            model=model_cfg.root_model,
            temperature=model_cfg.temperature
        )
        self.sub_llm = ChatOpenAI(
            model=model_cfg.sub_model,
            temperature=model_cfg.temperature
        )

        # Create agents
        base_tools = self.tool_factory.create_base_tools()
        self.sub_agent = create_deep_agent(model=self.sub_llm, tools=base_tools, name="rlm-sub-agent")

        subcall_tool = self.tool_factory.create_subcall_tool(self.sub_agent)
        root_tools = base_tools + [subcall_tool]
        self.root_agent = create_deep_agent(model=self.root_llm, tools=root_tools, name="rlm-root-agent")

        # Set system message
        self.system_message = system_message or self._default_system_message()

    def _default_system_message(self) -> str:
        """Generate the default system message."""
        return f"""You are an RLM (Recursive Language Model) analyzing {self.document_description}.

CRITICAL: You CANNOT see the document directly. It is stored in an environment variable.
The document has {len(self.context):,} characters total.

Your available tools:
- context_info(): Get document statistics and find section headers
- peek(start, end): View a slice of the document by character indices
- find_regex(pattern): Search for patterns and see matches with context
- subcall(start, end, question): Delegate focused analysis of a region to a cheaper sub-agent

Strategy tips:
1. Start with context_info() to understand the document structure
2. Use find_regex() to locate relevant sections by keywords
3. Use peek() to read specific regions you've identified
4. Use subcall() to delegate detailed analysis of specific sections

Build your answer incrementally. Be systematic and thorough."""

    def query(self, question: str, verbose: bool = False) -> str:
        """
        Run a query against the document.

        Args:
            question: The question to answer about the document
            verbose: If True, print intermediate steps

        Returns:
            The agent's final response
        """
        messages = [
            ("system", self.system_message),
            ("user", question)
        ]

        final_response = ""
        for chunk in self.root_agent.stream(
            {"messages": messages},
            stream_mode="values",
        ):
            if "messages" in chunk:
                msg = chunk["messages"][-1]
                if verbose and hasattr(msg, 'pretty_print'):
                    msg.pretty_print()
                if hasattr(msg, 'content') and msg.type == "ai":
                    final_response = msg.content

        return final_response

    def stream(
        self,
        question: str,
        include_tool_calls: bool = False
    ) -> Iterator[dict]:
        """
        Stream the agent's response.

        Args:
            question: The question to answer about the document
            include_tool_calls: If True, yield tool call information

        Yields:
            Dictionaries with 'type' and 'content' keys
        """
        messages = [
            ("system", self.system_message),
            ("user", question)
        ]

        for chunk in self.root_agent.stream(
            {"messages": messages},
            stream_mode="values",
        ):
            if "messages" in chunk:
                msg = chunk["messages"][-1]

                if msg.type == "ai" and hasattr(msg, 'content') and msg.content:
                    yield {"type": "response", "content": msg.content}
                elif include_tool_calls and msg.type == "tool":
                    yield {"type": "tool_result", "content": msg.content}

    # -------------------------------------------------------------------------
    # Factory Methods
    # -------------------------------------------------------------------------

    @classmethod
    def from_url(
        cls,
        url: str,
        config: AgentConfig | None = None,
        system_message: str | None = None,
        document_description: str | None = None,
        timeout: int = 30,
    ) -> RLMAgent:
        """
        Create an RLM agent from a URL.

        Args:
            url: URL to fetch content from
            config: Agent configuration
            system_message: Custom system message
            document_description: Description for the system prompt
            timeout: Request timeout in seconds
        """
        fetcher = URLFetcher(url, timeout)
        context = fetcher.fetch()
        desc = document_description or f"content from {url}"
        return cls(context, config, system_message, desc)

    @classmethod
    def from_file(
        cls,
        path: str | Path,
        config: AgentConfig | None = None,
        system_message: str | None = None,
        document_description: str | None = None,
        encoding: str = "utf-8",
    ) -> RLMAgent:
        """
        Create an RLM agent from a file.

        Args:
            path: Path to the file
            config: Agent configuration
            system_message: Custom system message
            document_description: Description for the system prompt
            encoding: File encoding
        """
        fetcher = FileFetcher(path, encoding)
        context = fetcher.fetch()
        desc = document_description or f"content from {Path(path).name}"
        return cls(context, config, system_message, desc)

    @classmethod
    def from_text(
        cls,
        text: str,
        config: AgentConfig | None = None,
        system_message: str | None = None,
        document_description: str = "a document",
    ) -> RLMAgent:
        """
        Create an RLM agent from raw text.

        Args:
            text: The document text
            config: Agent configuration
            system_message: Custom system message
            document_description: Description for the system prompt
        """
        return cls(text, config, system_message, document_description)

    # -------------------------------------------------------------------------
    # Properties
    # -------------------------------------------------------------------------

    @property
    def context_length(self) -> int:
        """Return the length of the context in characters."""
        return len(self.context)

    @property
    def context_lines(self) -> int:
        """Return the number of lines in the context."""
        return len(self.context.split('\n'))


# =============================================================================
# Convenience Functions
# =============================================================================

def create_agent_from_url(url: str, **kwargs) -> RLMAgent:
    """Convenience function to create an agent from a URL."""
    return RLMAgent.from_url(url, **kwargs)


def create_agent_from_file(path: str | Path, **kwargs) -> RLMAgent:
    """Convenience function to create an agent from a file."""
    return RLMAgent.from_file(path, **kwargs)


def create_agent_from_text(text: str, **kwargs) -> RLMAgent:
    """Convenience function to create an agent from text."""
    return RLMAgent.from_text(text, **kwargs)


# =============================================================================
# Demo / Example Usage
# =============================================================================

DEMO_QUERIES = {
    "benchmarks": (
        "What benchmarks were used to evaluate RLMs? For each benchmark, explain "
        "what it tests and summarize the key results comparing RLMs to baselines."
    ),
    "strategies": (
        "What emergent strategies did the authors observe RLMs using when analyzing context? "
        "List each strategy with a brief description of how it works."
    ),
    "limitations": (
        "What are the stated limitations of RLMs according to this paper? "
        "Also identify any implicit limitations you can infer from the methodology."
    ),
    "vs_agents": (
        "How do RLMs differ from traditional agent architectures like ReAct? "
        "What is the key philosophical difference in how they approach context decomposition?"
    ),
    "cost_analysis": (
        "Analyze the cost-effectiveness of RLMs based on the experimental results. "
        "Compare API costs between RLM approaches and baseline models."
    ),
    "future_work": (
        "What future directions do the authors suggest for RLM research? "
        "What aspects do they think could be improved or extended?"
    ),
}


def run_demo(query_name: str = "benchmarks", verbose: bool = True):
    """
    Run a demo query against the RLM blog post.

    Args:
        query_name: One of the keys in DEMO_QUERIES
        verbose: If True, print intermediate steps
    """
    if query_name not in DEMO_QUERIES:
        print(f"Unknown query. Choose from: {list(DEMO_QUERIES.keys())}")
        return

    print("Fetching RLM blog post...")
    agent = RLMAgent.from_url(
        "https://alexzhang13.github.io/blog/2025/rlm/",
        document_description="a blog post about Recursive Language Models (RLMs)"
    )
    print(f"Context loaded: {agent.context_length:,} characters")

    question = DEMO_QUERIES[query_name]
    print(f"\n{'='*60}")
    print(f"QUERY: {query_name}")
    print(f"{'='*60}")
    print(f"\n{question}\n")
    print(f"{'='*60}\n")

    response = agent.query(question, verbose=verbose)

    if not verbose:
        print("\nFINAL RESPONSE:")
        print(response)

    return response


if __name__ == "__main__":
    print("\n" + "="*60)
    print("RLM Agent Module")
    print("="*60)
    print("\nUsage examples:")
    print("""
    # From URL
    agent = RLMAgent.from_url("https://example.com/article")

    # From file
    agent = RLMAgent.from_file("document.txt")

    # From text
    agent = RLMAgent.from_text("Your document content here...")

    # Query the document
    response = agent.query("What are the main findings?")

    # With verbose output
    response = agent.query("Summarize the methodology", verbose=True)

    # Stream responses
    for chunk in agent.stream("What is the conclusion?"):
        print(chunk["content"])
    """)

    print("\nDemo queries available:")
    for name, q in DEMO_QUERIES.items():
        print(f"  - {name}: {q[:50]}...")

    print("\nRun demo with: run_demo('benchmarks')")

### Run V2 Demo

Fetch the RLM blog post and run the benchmarks query.

In [ ]:
run_demo('benchmarks')

## V3 — Custom LangGraph Implementation

Builds the RLM pattern with explicit `StateGraph` nodes and edges, giving full control over the agent loop, state management, and recursive delegation. Includes a `SubAgentGraph` for depth-limited sub-calls.

In [ ]:
"""
-----------------------------------------------------------------------------
V3 of the RLM-Style ReAct Agent Demo
-----------------------------------------------------------------------------
RLM (Recursive Language Model) - Custom LangGraph Implementation

This implements the RLM pattern with explicit graph structure, giving you
full control over the agent loop, state management, and recursive delegation.

Key concepts:
- State: Tracks messages, context, and recursion depth
- Nodes: route -> call_model -> call_tools (loop) or respond
- Subgraph: Separate graph for sub-agent with depth limiting
- Conditional edges: Control flow based on tool calls

Usage:
    from rlm_langgraph import RLMGraph

    graph = RLMGraph.from_url("https://example.com/article")
    response = graph.invoke("What are the main findings?")

    # Or stream
    for event in graph.stream("Summarize the methodology"):
        print(event)
"""
from __future__ import annotations

import re
import operator
from dataclasses import dataclass, field
from pathlib import Path
from typing import Annotated, Literal, TypedDict, Sequence, Any

import requests
from bs4 import BeautifulSoup
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool, BaseTool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class RLMConfig:
    """Configuration for the RLM graph."""
    root_model: str = "gpt-4.1-mini"
    sub_model: str = "gpt-4.1-nano"
    temperature: float = 0.0
    max_recursion_depth: int = 3  # Max depth for nested subcalls
    max_iterations: int = 15  # Max tool-call loops per agent
    max_regex_hits: int = 15
    context_window: int = 80  # chars around regex matches
    langgraph_recursion_limit: int = 1000  # LangGraph's internal recursion limit


# =============================================================================
# State Definitions
# =============================================================================

class AgentState(TypedDict):
    """State for the main RLM agent."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    context: str  # The document being analyzed
    iteration: int
    max_iterations: int


class SubAgentState(TypedDict):
    """State for the sub-agent (delegated analysis)."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    context: str
    snippet: str  # The specific region being analyzed
    snippet_start: int
    snippet_end: int
    iteration: int
    max_iterations: int
    depth: int  # Recursion depth


# =============================================================================
# Tool Definitions (as functions that take context)
# =============================================================================

def create_peek_tool(context: str) -> BaseTool:
    """Create a peek tool bound to a context."""

    @tool
    def peek(start: int, end: int) -> str:
        """
        Return a slice of the context document.

        Args:
            start: Starting character index (0-based)
            end: Ending character index (exclusive)
        """
        start = max(0, start)
        end = max(start, min(len(context), end))
        snippet = context[start:end]
        return f"[Characters {start}-{end} of {len(context)} total]\n\n{snippet}"

    return peek


def create_find_regex_tool(context: str, max_hits: int = 15, window: int = 80) -> BaseTool:
    """Create a regex search tool bound to a context."""

    @tool
    def find_regex(pattern: str, max_results: int | None = None) -> str:
        """
        Search for a regex pattern in the document.
        Returns match positions with surrounding context.

        Args:
            pattern: Regular expression pattern to search for
            max_results: Maximum matches to return (default 15)
        """
        limit = max_results or max_hits
        hits = []

        try:
            for m in re.finditer(pattern, context, flags=re.MULTILINE | re.IGNORECASE):
                s, e = m.start(), m.end()
                ctx_start = max(0, s - window)
                ctx_end = min(len(context), e + window)
                preview = context[ctx_start:ctx_end].replace('\n', ' ')

                rel_start = s - ctx_start
                rel_end = e - ctx_start
                marked = f"...{preview[:rel_start]}>>>{preview[rel_start:rel_end]}<<<{preview[rel_end:]}..."

                hits.append(f"[{s}:{e}] {marked}")
                if len(hits) >= limit:
                    break
        except re.error as err:
            return f"REGEX_ERROR: {err}"

        return '\n\n'.join(hits) if hits else "NO_MATCHES"

    return find_regex


def create_context_info_tool(context: str) -> BaseTool:
    """Create a context info tool bound to a context."""

    @tool
    def context_info() -> str:
        """
        Get document statistics and structural overview.
        Returns total length, line count, and potential section headers.
        """
        lines = context.split('\n')

        keywords = [
            'result', 'model', 'conclusion', 'introduction', 'method',
            'experiment', 'related', 'discussion', 'limitation',
            'benchmark', 'setup', 'abstract', 'summary', 'overview',
            'background', 'analysis', 'evaluation', 'appendix'
        ]

        sections = []
        for i, line in enumerate(lines):
            if 3 < len(line) < 100 and not line.endswith('.') and not line.startswith('-'):
                if any(kw in line.lower() for kw in keywords):
                    sections.append(f"  Line {i}: {line[:60]}...")

        return (
            f"Document Statistics:\n"
            f"  Total characters: {len(context):,}\n"
            f"  Total lines: {len(lines):,}\n"
            f"\nPotential section headers found:\n" + '\n'.join(sections[:20])
        )

    return context_info


# =============================================================================
# Sub-Agent Graph (for delegated analysis)
# =============================================================================

class SubAgentGraph:
    """
    A sub-graph for focused analysis of document sections.
    This is invoked by the 'subcall' tool in the main graph.
    """

    def __init__(self, context: str, config: RLMConfig):
        self.context = context
        self.config = config
        self.llm = ChatOpenAI(
            model=config.sub_model,
            temperature=config.temperature
        )

        # Sub-agent tools (no subcall to prevent infinite recursion)
        self.tools = [
            create_peek_tool(context),
            create_find_regex_tool(context, config.max_regex_hits, config.context_window),
            create_context_info_tool(context),
        ]
        self.llm_with_tools = self.llm.bind_tools(self.tools)
        self.tool_node = ToolNode(self.tools)

        self.graph = self._build_graph()

    def _build_graph(self) -> StateGraph:
        """Build the sub-agent graph."""

        def call_model(state: SubAgentState) -> dict:
            """Invoke the LLM."""
            messages = state["messages"]
            response = self.llm_with_tools.invoke(messages)
            return {
                "messages": [response],
                "iteration": state["iteration"] + 1
            }

        def should_continue(state: SubAgentState) -> Literal["tools", "end"]:
            """Decide whether to continue tool calling or end."""
            messages = state["messages"]
            last_message = messages[-1]

            # Check iteration limit
            if state["iteration"] >= state["max_iterations"]:
                return "end"

            # Check for tool calls
            if hasattr(last_message, "tool_calls") and last_message.tool_calls:
                return "tools"

            return "end"

        # Build graph
        builder = StateGraph(SubAgentState)

        builder.add_node("agent", call_model)
        builder.add_node("tools", self.tool_node)

        builder.add_edge(START, "agent")
        builder.add_conditional_edges(
            "agent",
            should_continue,
            {"tools": "tools", "end": END}
        )
        builder.add_edge("tools", "agent")

        return builder.compile()

    def invoke(
        self,
        snippet_start: int,
        snippet_end: int,
        question: str,
        depth: int = 1
    ) -> str:
        """
        Invoke the sub-agent on a document snippet.

        Args:
            snippet_start: Start index of the region
            snippet_end: End index of the region
            question: Question to answer about this region
            depth: Current recursion depth

        Returns:
            The sub-agent's analysis
        """
        snippet_start = max(0, snippet_start)
        snippet_end = min(len(self.context), snippet_end)
        snippet = self.context[snippet_start:snippet_end]

        system_prompt = (
            f"You are analyzing a snippet from a larger document.\n\n"
            f"SNIPPET (characters {snippet_start}-{snippet_end} of {len(self.context)} total):\n"
            f"```\n{snippet}\n```\n\n"
            f"You have tools to explore the FULL document if needed:\n"
            f"- peek(start, end): View any region of the full document\n"
            f"- find_regex(pattern): Search the full document\n"
            f"- context_info(): Get document structure\n\n"
            f"Focus on answering the question based primarily on the snippet, "
            f"but use tools if you need additional context."
        )

        initial_state: SubAgentState = {
            "messages": [
                SystemMessage(content=system_prompt),
                HumanMessage(content=question)
            ],
            "context": self.context,
            "snippet": snippet,
            "snippet_start": snippet_start,
            "snippet_end": snippet_end,
            "iteration": 0,
            "max_iterations": self.config.max_iterations,
            "depth": depth,
        }

        result = self.graph.invoke(
            initial_state,
            {"recursion_limit": self.config.langgraph_recursion_limit}
        )

        # Extract final response
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                return msg.content

        return "Sub-agent produced no response"


# =============================================================================
# Main RLM Graph
# =============================================================================

class RLMGraph:
    """
    The main RLM (Recursive Language Model) graph.

    Implements a ReAct-style agent with:
    - peek: View document regions
    - find_regex: Search the document
    - context_info: Get document structure
    - subcall: Delegate analysis to a cheaper sub-agent
    """

    def __init__(
        self,
        context: str,
        config: RLMConfig | None = None,
        system_message: str | None = None,
        document_description: str = "a document",
    ):
        self.context = context
        self.config = config or RLMConfig()
        self.document_description = document_description

        # Initialize LLM
        self.llm = ChatOpenAI(
            model=self.config.root_model,
            temperature=self.config.temperature
        )

        # Create sub-agent graph
        self.sub_agent = SubAgentGraph(context, self.config)

        # Create tools (including subcall)
        self.tools = self._create_tools()
        self.llm_with_tools = self.llm.bind_tools(self.tools)

        # Custom tool node that handles subcall specially
        self.tool_executor = self._create_tool_executor()

        # System message
        self.system_message = system_message or self._default_system_message()

        # Build the graph
        self.graph = self._build_graph()

    def _create_tools(self) -> list[BaseTool]:
        """Create all tools for the root agent."""
        context = self.context
        config = self.config
        sub_agent = self.sub_agent

        # Basic tools
        tools = [
            create_peek_tool(context),
            create_find_regex_tool(context, config.max_regex_hits, config.context_window),
            create_context_info_tool(context),
        ]

        # Subcall tool
        @tool
        def subcall(snippet_start: int, snippet_end: int, question: str) -> str:
            """
            Delegate analysis of a specific document region to a sub-agent.
            The sub-agent uses a cheaper model for focused analysis.

            Args:
                snippet_start: Start index of the region to analyze
                snippet_end: End index of the region to analyze
                question: The specific question to answer about this region
            """
            return sub_agent.invoke(snippet_start, snippet_end, question)

        tools.append(subcall)
        return tools

    def _create_tool_executor(self):
        """Create a tool executor node."""
        return ToolNode(self.tools)

    def _default_system_message(self) -> str:
        """Generate the default system message."""
        return f"""You are an RLM (Recursive Language Model) analyzing {self.document_description}.

CRITICAL: You CANNOT see the document directly. It is stored externally.
The document has {len(self.context):,} characters total.

Your available tools:
- context_info(): Get document statistics and find section headers
- peek(start, end): View a slice of the document by character indices
- find_regex(pattern): Search for patterns and see matches with context
- subcall(start, end, question): Delegate focused analysis to a cheaper sub-agent

Strategy for effective analysis:
1. Start with context_info() to understand document structure
2. Use find_regex() to locate relevant sections by keywords
3. Use peek() to read specific regions you've identified
4. Use subcall() to delegate detailed analysis of specific sections

Build your answer incrementally. Be systematic and thorough.
When you have enough information, provide a comprehensive answer."""

    def _build_graph(self) -> StateGraph:
        """Build the main agent graph."""

        def call_model(state: AgentState) -> dict:
            """Invoke the LLM with current messages."""
            messages = state["messages"]
            response = self.llm_with_tools.invoke(messages)
            return {
                "messages": [response],
                "iteration": state["iteration"] + 1
            }

        def should_continue(state: AgentState) -> Literal["tools", "end"]:
            """Decide whether to continue or end."""
            messages = state["messages"]
            last_message = messages[-1]

            # Check iteration limit
            if state["iteration"] >= state["max_iterations"]:
                return "end"

            # Check for tool calls
            if hasattr(last_message, "tool_calls") and last_message.tool_calls:
                return "tools"

            return "end"

        # Build graph
        builder = StateGraph(AgentState)

        builder.add_node("agent", call_model)
        builder.add_node("tools", self.tool_executor)

        builder.add_edge(START, "agent")
        builder.add_conditional_edges(
            "agent",
            should_continue,
            {"tools": "tools", "end": END}
        )
        builder.add_edge("tools", "agent")

        return builder.compile()

    def invoke(self, question: str) -> str:
        """
        Run a query against the document.

        Args:
            question: The question to answer

        Returns:
            The agent's final response
        """
        initial_state: AgentState = {
            "messages": [
                SystemMessage(content=self.system_message),
                HumanMessage(content=question)
            ],
            "context": self.context,
            "iteration": 0,
            "max_iterations": self.config.max_iterations,
        }

        result = self.graph.invoke(
            initial_state,
            {"recursion_limit": self.config.langgraph_recursion_limit}
        )

        # Extract final response
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                return msg.content

        return "Agent produced no response"

    def stream(self, question: str):
        """
        Stream the agent's execution.

        Args:
            question: The question to answer

        Yields:
            Events from the graph execution
        """
        initial_state: AgentState = {
            "messages": [
                SystemMessage(content=self.system_message),
                HumanMessage(content=question)
            ],
            "context": self.context,
            "iteration": 0,
            "max_iterations": self.config.max_iterations,
        }

        for event in self.graph.stream(
            initial_state,
            stream_mode="updates",
            config={"recursion_limit": self.config.langgraph_recursion_limit}
        ):
            yield event

    def stream_events(self, question: str):
        """
        Stream detailed events from the agent.

        Args:
            question: The question to answer

        Yields:
            Detailed event dictionaries
        """
        initial_state: AgentState = {
            "messages": [
                SystemMessage(content=self.system_message),
                HumanMessage(content=question)
            ],
            "context": self.context,
            "iteration": 0,
            "max_iterations": self.config.max_iterations,
        }

        for event in self.graph.stream(
            initial_state,
            stream_mode="values",
            config={"recursion_limit": self.config.langgraph_recursion_limit}
        ):
            messages = event.get("messages", [])
            if messages:
                last_msg = messages[-1]

                if isinstance(last_msg, AIMessage):
                    if last_msg.tool_calls:
                        yield {
                            "type": "tool_calls",
                            "calls": [
                                {"name": tc["name"], "args": tc["args"]}
                                for tc in last_msg.tool_calls
                            ]
                        }
                    elif last_msg.content:
                        yield {
                            "type": "response",
                            "content": last_msg.content
                        }
                elif isinstance(last_msg, ToolMessage):
                    yield {
                        "type": "tool_result",
                        "name": last_msg.name,
                        "content": last_msg.content[:500] + "..." if len(last_msg.content) > 500 else last_msg.content
                    }

    # -------------------------------------------------------------------------
    # Factory Methods
    # -------------------------------------------------------------------------

    @classmethod
    def from_url(
        cls,
        url: str,
        config: RLMConfig | None = None,
        system_message: str | None = None,
        document_description: str | None = None,
        timeout: int = 30,
    ) -> RLMGraph:
        """Create an RLM graph from a URL."""
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')
        for element in soup(['script', 'style', 'nav', 'header', 'footer']):
            element.decompose()

        main_content = soup.find('main') or soup.find('article') or soup.body
        if main_content:
            text = main_content.get_text(separator='\n', strip=True)
        else:
            text = soup.get_text(separator='\n', strip=True)

        lines = [line.strip() for line in text.split('\n') if line.strip()]
        context = '\n'.join(lines)

        desc = document_description or f"content from {url}"
        return cls(context, config, system_message, desc)

    @classmethod
    def from_file(
        cls,
        path: str | Path,
        config: RLMConfig | None = None,
        system_message: str | None = None,
        document_description: str | None = None,
        encoding: str = "utf-8",
    ) -> RLMGraph:
        """Create an RLM graph from a file."""
        path = Path(path)
        context = path.read_text(encoding=encoding)
        desc = document_description or f"content from {path.name}"
        return cls(context, config, system_message, desc)

    @classmethod
    def from_text(
        cls,
        text: str,
        config: RLMConfig | None = None,
        system_message: str | None = None,
        document_description: str = "a document",
    ) -> RLMGraph:
        """Create an RLM graph from raw text."""
        return cls(text, config, system_message, document_description)

    # -------------------------------------------------------------------------
    # Visualization
    # -------------------------------------------------------------------------

    def get_graph_diagram(self) -> str:
        """Get a Mermaid diagram of the graph structure."""
        return """
graph TD
    START([Start]) --> agent[Agent Node<br/>Call LLM]
    agent -->|has tool calls| tools[Tool Node<br/>Execute Tools]
    agent -->|no tool calls| END([End])
    tools --> agent

    subgraph "Tools Available"
        peek[peek - View document slice]
        find_regex[find_regex - Search patterns]
        context_info[context_info - Document stats]
        subcall[subcall - Delegate to sub-agent]
    end

    tools -.-> peek
    tools -.-> find_regex
    tools -.-> context_info
    tools -.-> subcall

    subgraph "Sub-Agent Graph"
        sub_start([Start]) --> sub_agent[Sub-Agent<br/>Cheaper LLM]
        sub_agent -->|has tool calls| sub_tools[Tool Node]
        sub_agent -->|no tool calls| sub_end([End])
        sub_tools --> sub_agent
    end

    subcall -.-> sub_start
"""

    @property
    def context_length(self) -> int:
        """Return document length in characters."""
        return len(self.context)


# =============================================================================
# Advanced: Custom Graph Builder
# =============================================================================

class RLMGraphBuilder:
    """
    Builder for creating customized RLM graphs.

    Allows adding custom tools, nodes, and edges.
    """

    def __init__(self, context: str, config: RLMConfig | None = None):
        self.context = context
        self.config = config or RLMConfig()
        self.custom_tools: list[BaseTool] = []
        self.custom_nodes: dict[str, Any] = {}
        self.system_message: str | None = None
        self.document_description = "a document"

    def add_tool(self, tool: BaseTool) -> RLMGraphBuilder:
        """Add a custom tool to the agent."""
        self.custom_tools.append(tool)
        return self

    def set_system_message(self, message: str) -> RLMGraphBuilder:
        """Set a custom system message."""
        self.system_message = message
        return self

    def set_description(self, description: str) -> RLMGraphBuilder:
        """Set the document description."""
        self.document_description = description
        return self

    def with_models(self, root: str, sub: str) -> RLMGraphBuilder:
        """Configure the models to use."""
        self.config.root_model = root
        self.config.sub_model = sub
        return self

    def with_max_iterations(self, n: int) -> RLMGraphBuilder:
        """Set maximum iterations (tool-call loops per agent)."""
        self.config.max_iterations = n
        return self

    def with_recursion_limit(self, limit: int) -> RLMGraphBuilder:
        """Set LangGraph's recursion limit (max graph steps)."""
        self.config.langgraph_recursion_limit = limit
        return self

    def with_max_depth(self, depth: int) -> RLMGraphBuilder:
        """Set maximum depth for nested subcalls."""
        self.config.max_recursion_depth = depth
        return self

    def build(self) -> RLMGraph:
        """Build the configured RLM graph."""
        graph = RLMGraph(
            self.context,
            self.config,
            self.system_message,
            self.document_description
        )

        # Add custom tools
        if self.custom_tools:
            graph.tools.extend(self.custom_tools)
            graph.llm_with_tools = graph.llm.bind_tools(graph.tools)
            graph.tool_executor = ToolNode(graph.tools)
            # Rebuild graph with new tools
            graph.graph = graph._build_graph()

        return graph


# =============================================================================
# Demo
# =============================================================================

DEMO_QUERIES = {
    "benchmarks": "What benchmarks were used to evaluate RLMs? Summarize the key results.",
    "strategies": "What emergent strategies did the authors observe RLMs using?",
    "limitations": "What are the stated limitations of RLMs?",
    "vs_agents": "How do RLMs differ from traditional agent architectures like ReAct?",
}


def run_demo(query_name: str = "benchmarks", verbose: bool = True):
    """Run a demo query against the RLM blog post."""
    if query_name not in DEMO_QUERIES:
        print(f"Choose from: {list(DEMO_QUERIES.keys())}")
        return

    print("Fetching RLM blog post...")
    graph = RLMGraph.from_url(
        "https://alexzhang13.github.io/blog/2025/rlm/",
        document_description="a blog post about Recursive Language Models"
    )
    print(f"Context loaded: {graph.context_length:,} characters\n")

    question = DEMO_QUERIES[query_name]
    print(f"Query: {question}\n")
    print("="*60)

    if verbose:
        for event in graph.stream_events(question):
            if event["type"] == "tool_calls":
                for call in event["calls"]:
                    print(f"\n🔧 Tool: {call['name']}")
                    print(f"   Args: {call['args']}")
            elif event["type"] == "tool_result":
                print(f"\n📋 Result from {event['name']}:")
                print(f"   {event['content'][:200]}...")
            elif event["type"] == "response":
                print(f"\n💬 Response:\n{event['content']}")
    else:
        response = graph.invoke(question)
        print(response)


if __name__ == "__main__":
    print("="*60)
    print("RLM LangGraph Implementation")
    print("="*60)
    print("""
Usage:
    # Basic
    graph = RLMGraph.from_url("https://example.com/doc")
    response = graph.invoke("What are the main findings?")

    # With streaming
    for event in graph.stream_events("Summarize"):
        print(event)

    # Using builder
    graph = (RLMGraphBuilder(text)
        .with_models("gpt-4o", "gpt-4o-mini")
        .set_description("research paper")
        .add_tool(my_custom_tool)
        .build())

    # View graph structure
    print(graph.get_graph_diagram())
""")

    print("\nDemo queries:", list(DEMO_QUERIES.keys()))
    print("\nRun: run_demo('benchmarks')")

### Run V3 Demo

Test the custom LangGraph implementation with the benchmarks query.

In [ ]:
run_demo('benchmarks')

## V4 — Minimal REPL-Based RLM

A lightweight implementation that uses LLM-generated Python code executed in a sandboxed REPL. Demonstrates the core RLM loop: **plan → execute → observe → repeat** with recursive `sub_call` delegation.

In [ ]:
import operator
import re
import sys
import io
from contextlib import redirect_stdout
from typing import Annotated, List, Optional, TypedDict, Union, Dict, Any

# Dependencies: pip install langgraph langchain-openai langchain-core
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# --- 1. State Definition ---
class RLMState(TypedDict):
    """
    The memory of the agent.
    Crucially, 'context_handle' is a POINTER, not the full text.
    """
    query: str
    context_handle: str         # Path to the file or ID of the data
    repl_history: Annotated[List[str], operator.add]
    depth: int                  # Safety counter for recursion
    final_answer: Optional[str]
    messages: List[Any]         # To track chat history for the LLM

# --- 2. The Environment (The "Heavy" Lifting) ---
# This simulates the "Environment" that holds the actual data.
# The LLM never sees the full content of 'data_store' directly.

with open("paper.txt", "r", encoding="utf-8") as f:
    file_contents = f.read()

DATA_STORE = {
    "doc_1.txt": file_contents,
    # ... imagine 10GB of text here ...
}

def load_context(handle: str) -> str:
    """Mock loading a heavy file."""
    return DATA_STORE.get(handle, "Error: File not found.")

# --- 3. Tools available in the REPL ---

def grep(pattern: str, handle: str) -> str:
    """Search for a regex pattern in the context."""
    content = load_context(handle)
    matches = re.findall(pattern, content, re.IGNORECASE)
    if not matches:
        return "No matches found."
    return f"Found {len(matches)} matches: {matches[:5]}..." # Truncate for LLM view

def peek(start: int, length: int, handle: str) -> str:
    """Read a specific chunk of the file."""
    content = load_context(handle)
    if start >= len(content):
        return "EOF"
    return content[start:start+length]

def sub_call(query: str, virtual_handle_content: str, current_depth: int) -> str:
    """
    The Recursive Step.
    Spawns a NEW RLM agent to process a specific sub-problem.
    """
    print(f"    [Recursion Depth {current_depth+1}] Spawning sub-agent for: '{query}'")

    # In a real system, we'd save the virtual content to a new handle
    temp_handle = f"temp_{hash(virtual_handle_content)}"
    DATA_STORE[temp_handle] = virtual_handle_content

    # Invoke the graph recursively
    result = run_rlm_agent(query, temp_handle, max_depth=3, current_depth=current_depth + 1)
    return f"Sub-agent result: {result}"

# --- 4. Nodes ---

# Initialize the LLM (Requires OPENAI_API_KEY env var)
# We use a lower temperature for deterministic code generation
# Root model for main planning
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Fast model for sub-agents
model_fast = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

def planner_node(state: RLMState, use_fast_model: bool = False):
    """
    The Brain (LLM).
    Decides whether to grep, peek, recurse, or answer.
    """
    query = state['query']
    handle = state['context_handle']
    history = "\n".join(state['repl_history'])

    # Use fast model for deeper recursion, root model for top-level
    current_model = model_fast if use_fast_model or state['depth'] > 0 else model

    system_prompt = f"""You are a Recursive Language Model (RLM).
You have a query: "{query}"
You have a context file located at: "{handle}"

You CANNOT see the file content directly. You must explore it using tools.

Tools available:
1. grep(pattern, handle) - Search for text. Returns a summary of matches.
2. peek(start, length, handle) - Read text. Returns the string.
3. sub_call(query, specific_content, depth) - Delegate to a sub-agent.
4. FINAL(answer) - Output the final answer.

Current Execution History:
{history if history else "No actions taken yet."}

INSTRUCTIONS:
- Return ONLY valid Python code to execute the next step.
- Do not add markdown backticks.
- Use the variable 'handle' for the file path.
- Always wrap tool calls in print() to see the output in the history.
- Example: print(grep("revenue", handle))
"""

    messages = [SystemMessage(content=system_prompt)]

    # Call the actual LLM
    response = current_model.invoke(messages)

    # Basic cleanup of markdown code blocks if the LLM adds them
    code = response.content.replace("```python", "").replace("```", "").strip()

    return {"messages": [AIMessage(content=code)]}

def executor_node(state: RLMState):
    """
    The Hands (REPL).
    Executes the code generated by the Planner.
    """
    last_message = state['messages'][-1].content
    handle = state['context_handle']
    depth = state['depth']

    # Check for termination
    if "FINAL(" in last_message:
        # Extract content between FINAL( and the last )
        try:
            answer = last_message.split("FINAL(", 1)[1].rsplit(")", 1)[0]
            # Handle quotes if present
            if (answer.startswith('"') and answer.endswith('"')) or (answer.startswith("'") and answer.endswith("'")):
                answer = answer[1:-1]
            return {"final_answer": answer}
        except IndexError:
            return {"repl_history": ["Error: Malformed FINAL() call."]}

    # Prepare local environment
    local_env = {
        "grep": grep,
        "peek": peek,
        "sub_call": lambda q, c: sub_call(q, c, depth),
        "handle": handle,
        # 'print' is handled by redirect_stdout below
    }

    result = ""
    try:
        # Capture stdout to simulate REPL output
        f = io.StringIO()
        with redirect_stdout(f):
            # DANGEROUS: exec() is used for demonstration.
            # In production, use e2b or a dockerized sandbox.
            exec(last_message, {}, local_env)

        output = f.getvalue().strip()
        result = f"Code: {last_message}\nOutput: {output}"

    except Exception as e:
        result = f"Code: {last_message}\nError: {str(e)}"

    return {"repl_history": [result]}

def should_continue(state: RLMState):
    if state.get("final_answer"):
        return "end"
    if state["depth"] > 5: # Hard stop
        return "end"
    return "continue"

# --- 5. Graph Construction ---

workflow = StateGraph(RLMState)

workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)

workflow.set_entry_point("planner")

workflow.add_conditional_edges(
    "executor",
    should_continue,
    {
        "continue": "planner",
        "end": END
    }
)

workflow.add_edge("planner", "executor")

app = workflow.compile()

# Wrapper to allow the 'sub_call' tool to invoke the app
def run_rlm_agent(query, handle, max_depth=3, current_depth=0):
    initial_state = {
        "query": query,
        "context_handle": handle,
        "repl_history": [],
        "depth": current_depth,
        "final_answer": None,
        "messages": []
    }

    # Run the graph
    final_state = app.invoke(initial_state, {"recursion_limit": 1000})
    return final_state.get("final_answer", "No answer found")

# --- 6. Execution ---

if __name__ == "__main__":
    # Initial state matching RLMState schema
    initial_state = {
        "query": "Which is generally done first, BFS or DFS?",
        "context_handle": "doc_1.txt",
        "repl_history": [],
        "depth": 0,
        "final_answer": None,
        "messages": []
    }

    print("--- Starting RLM DeepAgent Loop ---")

    # Stream with stream_mode="values" and pretty_print the last message
    for state in app.stream(initial_state, {"recursion_limit": 1000}, stream_mode="values"):
        print("\n--- State Update ---")

        # Pretty print the last message if messages exist
        if state.get("messages"):
            last_msg = state["messages"][-1]
            if hasattr(last_msg, "pretty_print"):
                last_msg.pretty_print()
            else:
                print(f"Last message: {last_msg}")

        # Print other relevant state info
        if state.get("repl_history"):
            print(f"REPL History: {state['repl_history'][-1] if state['repl_history'] else 'None'}")

        if state.get("final_answer"):
            print(f"\n=== FINAL ANSWER ===\n{state['final_answer']}")